# Camada B/C

Notebook de decisão. Ele não assume que a camada B serve nem que não serve: mede, corrige o que dá para corrigir e emite um veredito com critério explícito no fim.

## Objetivos

A execução do data wrangling deixou a camada B em dois estados diferentes, e é importante não confundi-los.

**O que já está bom.** Ativo total e patrimônio líquido com 100% de cobertura em todos os doze exercícios. Índice de Basileia com 96% de 2015 a 2024, e zero em 2014 apenas porque o relatório de Informações de Capital não existia antes de 2015. A fusão de rótulos resolveu o que parecia ser 52% de ausência.

**O que está quebrado.** O Lucro Líquido não foi localizado no mapeamento, e é dele que vêm os 52% de ausência em ROA e ROE. E o rótulo de resolução ficou zerado, porque a lista oficial do BACEN não estava em disco.

**O que é característica da base, não defeito.** O Índice de Capital Principal cai de 97% para 32% a partir de 2018, quando as instituições do segmento S5 deixam de apurá-lo por força da segmentação da Resolução 4.553. Isso vira indicador de ausência, não descarte.

Ao final, um critério objetivo separa três desfechos: a camada B vira estudo preditivo próprio, vira análise descritiva de contraponto, ou sai do artigo com uma frase em trabalhos futuros. A decisão sai dos números, não da vontade.

## 1. Configuração

In [1]:
!pip install -q pandas numpy pyarrow requests

In [2]:
import re
import json
import unicodedata
import warnings
from pathlib import Path
from datetime import datetime
from difflib import SequenceMatcher

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

RAIZ = Path(r"G:\Meu Drive\TCC\Damaris Rebeca")
DIR_BRUTO = RAIZ / "01_bruto"
DIR_RES = DIR_BRUTO / "bcb_resolucao"
DIR_INTER = RAIZ / "02_intermediario"
DIR_ANALITICO = RAIZ / "03_analitico"
DIR_MODELAGEM = RAIZ / "04_modelagem"
for d in (DIR_RES, DIR_INTER, DIR_MODELAGEM):
    d.mkdir(parents=True, exist_ok=True)

UF_RECORTE = "MG"
MIN_POSITIVOS_MODELAGEM = 30
SIMILARIDADE_MINIMA = 0.80

def normalizar(s):
    s = unicodedata.normalize("NFKD", str(s) if pd.notna(s) else "")
    s = s.encode("ascii", "ignore").decode("ascii")
    return " ".join(s.upper().split())

def limpar_rotulo(c):
    s = str(c).replace("\r", " ").split("\n")[0]
    s = re.sub(r"\(\s*[a-z]\d?\s*\)\s*=.*$", "", s, flags=re.I)
    s = re.sub(r"\(\s*[a-z]\d?\s*\)\s*$", "", s, flags=re.I)
    return re.sub(r"\s+", " ", s).strip()

b0 = pd.read_parquet(DIR_ANALITICO / "camada_b_bacen_painel.parquet")
print(f"painel bruto da entrega 2: {len(b0):,} linhas x {b0.shape[1]} colunas")
print("exercicios:", sorted(b0["ano"].unique()))

painel bruto da entrega 2: 16,552 linhas x 128 colunas
exercicios: [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


---
## 2. Que rótulos existem de fato

Antes de mapear qualquer coisa, listar o que o BACEN realmente publica. A busca é por palavra-chave nos rótulos já limpos, sem supor a redação exata. É isso que vai revelar sob que nome o Lucro Líquido está escondido.

In [4]:
CHAVES_B = {"ano", "data_base", "tipo_instituicao", "cod_inst", "instituicao", "tcb",
            "consolidacao", "controle", "uf", "cidade", "segmento", "fonte",
            "y_resolucao", "descartar_pos_evento"}

medidas_orig = [c for c in b0.columns if c not in CHAVES_B]
mapa_limpo = {c: limpar_rotulo(c) for c in medidas_orig}

grupos = {}
for orig, limpo in mapa_limpo.items():
    grupos.setdefault(limpo, []).append(orig)

print(f"{len(medidas_orig)} colunas originais | {len(grupos)} rotulos apos limpeza\n")

TERMOS = ["LUCRO", "RESULTADO", "PREJUIZO", "BASILEIA", "CAPITAL", "PATRIMONIO",
          "ATIVO", "CAPTAC", "CREDITO", "PROVIS", "RECEITA", "DESPESA"]

achados = {}
for termo in TERMOS:
    lista = sorted(r for r in grupos if termo in normalizar(r))
    achados[termo] = lista
    print(f"--- {termo} ({len(lista)})")
    for r in lista[:10]:
        print("   ", r[:88])
    if len(lista) > 10:
        print(f"    ... mais {len(lista)-10}")

114 colunas originais | 105 rotulos apos limpeza

--- LUCRO (4)
    Lucro Líquido
    Participação nos Lucros
    Resultado antes da Tributação, Lucro e Participação
    lucro_liquido
--- RESULTADO (9)
    Passivo Circulante e Exigível a Longo Prazo e Resultados de Exercícios Futuros
    Resultado Não Operacional
    Resultado Operacional
    Resultado antes da Tributação, Lucro e Participação
    Resultado de Intermediação Financeira
    Resultado de Operações de Câmbio
    Resultado de Participações
    Resultado de Provisão para Créditos de Difícil Liquidação
    Resultados de Exercícios Futuros
--- PREJUIZO (1)
    prejuizo
--- BASILEIA (2)
    basileia
    Índice de Basileia
--- CAPITAL (7)
    Capital Complementar
    Capital Nível II
    Capital Principal para Comparação com RWA
    Juros Sobre Capital Próprio
    capital_principal
    Índice de Capital Nível I
    Índice de Capital Principal
--- PATRIMONIO (5)
    Patrimônio Líquido
    Patrimônio de Referência Nível I para Com

In [5]:
linhas = []
for limpo, origens in grupos.items():
    serie = pd.to_numeric(b0[origens[0]], errors="coerce")
    for extra in origens[1:]:
        serie = serie.fillna(pd.to_numeric(b0[extra], errors="coerce"))
    por_ano = serie.groupby(b0["ano"]).apply(lambda x: x.notna().mean())
    linhas.append({"rotulo": limpo, "colunas_origem": len(origens),
                   "anos_com_dado": int((por_ano > 0.2).sum()),
                   "cobertura_geral": float(serie.notna().mean())})

catalogo = pd.DataFrame(linhas).sort_values(["anos_com_dado", "cobertura_geral"], ascending=False)
catalogo.to_csv(DIR_INTER / "catalogo_rotulos_ifdata.csv", index=False, encoding="utf-8-sig")

print("rotulos com cobertura em pelo menos 10 exercicios\n")
print(catalogo[catalogo["anos_com_dado"] >= 10].head(40).to_string(index=False))
print(f"\ncatalogo completo gravado em {DIR_INTER / 'catalogo_rotulos_ifdata.csv'}")

rotulos com cobertura em pelo menos 10 exercicios

                                                  rotulo  colunas_origem  anos_com_dado  cobertura_geral
                                                prejuizo               1             12           1.0000
                 Aplicações Interfinanceiras de Liquidez               1             11           0.9187
              Arrendamento Mercantil Líquido de Provisão               1             11           0.9187
                        Arrendamento Mercantil a Receber               1             11           0.9187
                                             Ativo Total               1             11           0.9187
                                    Ativo Total Ajustado               1             11           0.9187
                                               Captações               1             11           0.9187
                        Carteira de Crédito Classificada               1             11           0.9187
    

---
## 3. Mapeamento corrigido

Agora o mapeamento usa três níveis, nessa ordem: nome exato, nome exato de uma lista de sinônimos, e busca por todos os termos obrigatórios presentes no rótulo. Cada indicador reporta como foi encontrado, o que torna a escolha auditável em vez de mágica.

In [7]:
base = b0[[c for c in b0.columns if c in CHAVES_B]].copy()
for limpo, origens in grupos.items():
    if not limpo:
        continue
    serie = pd.to_numeric(b0[origens[0]], errors="coerce")
    for extra in origens[1:]:
        serie = serie.fillna(pd.to_numeric(b0[extra], errors="coerce"))
    base[limpo] = serie

rotulos = [c for c in base.columns if c not in CHAVES_B]
norm_rotulos = {c: normalizar(c) for c in rotulos}

MAPA = {
    "ativo_total": {"exatos": ["Ativo Total"], "termos": ["ATIVO", "TOTAL"]},
    "patrimonio_liquido": {"exatos": ["Patrimônio Líquido"], "termos": ["PATRIMONIO", "LIQUIDO"]},
    "lucro_liquido": {"exatos": ["Lucro Líquido"], "termos": ["LUCRO", "LIQUIDO"]},
    "resultado_antes_tributacao": {"exatos": ["Resultado antes da Tributação, Lucro e Participação"],
                                   "termos": ["RESULTADO", "TRIBUTACAO"]},
    "resultado_operacional": {"exatos": ["Resultado Operacional"], "termos": ["RESULTADO", "OPERACIONAL"]},
    "resultado_nao_operacional": {"exatos": ["Resultado Não Operacional"],
                                  "termos": ["RESULTADO", "NAO", "OPERACIONAL"]},
    "resultado_intermediacao": {"exatos": ["Resultado de Intermediação Financeira"],
                                "termos": ["INTERMEDIACAO"]},
    "provisao_credito": {"exatos": ["Resultado de Provisão para Créditos de Difícil Liquidação"],
                         "termos": ["PROVISAO", "CREDITOS"]},
    "participacao_lucros": {"exatos": ["Participação nos Lucros"], "termos": ["PARTICIPACAO", "LUCROS"]},
    "tributos": {"exatos": ["Imposto de Renda e Contribuição Social"], "termos": ["IMPOSTO"]},
    "captacoes": {"exatos": ["Captações"], "termos": ["CAPTAC"]},
    "carteira_credito": {"exatos": ["Carteira de Crédito Classificada"],
                         "termos": ["CARTEIRA", "CREDITO", "CLASSIFICADA"]},
    "basileia": {"exatos": ["Índice de Basileia"], "termos": ["INDICE", "BASILEIA"]},
    "capital_principal": {"exatos": ["Índice de Capital Principal"],
                          "termos": ["INDICE", "CAPITAL", "PRINCIPAL"]},
    "patrimonio_referencia": {"exatos": ["Patrimônio de Referência para Comparação com o RWA"],
                              "termos": ["PATRIMONIO", "REFERENCIA"]},
    "rwa": {"exatos": ["Ativos Ponderados pelo Risco (RWA)"], "termos": ["PONDERADOS", "RISCO"]},
    "passivo_exigivel": {"exatos": ["Passivo Circulante e Exigível a Longo Prazo"],
                         "termos": ["PASSIVO", "CIRCULANTE"]},
}

def localizar(cfg):
    for nome in cfg["exatos"]:
        alvo = normalizar(nome)
        for c, n in norm_rotulos.items():
            if n == alvo:
                return c, "nome exato"
    termos = [normalizar(t) for t in cfg["termos"]]
    candidatos = [c for c, n in norm_rotulos.items() if all(t in n for t in termos)]
    if candidatos:
        melhor = max(candidatos, key=lambda c: base[c].notna().mean())
        return melhor, "termos obrigatorios"
    return None, "nao encontrado"

encontrados, relatorio = {}, []
for destino, cfg in MAPA.items():
    col, como = localizar(cfg)
    encontrados[destino] = col
    cobertura = float(base[col].notna().mean()) if col else 0.0
    relatorio.append({"indicador": destino, "rotulo_no_ifdata": col or "", "criterio": como,
                      "cobertura": round(cobertura, 4)})
    if col:
        base[destino] = base[col]

rel = pd.DataFrame(relatorio)
print(rel.to_string(index=False))

faltando = rel[rel["rotulo_no_ifdata"] == ""]["indicador"].tolist()
if faltando:
    print("\nNAO LOCALIZADOS:", faltando)
    print("Procure o nome real em 02_intermediario/catalogo_rotulos_ifdata.csv e acrescente em MAPA['exatos'].")

                 indicador                                          rotulo_no_ifdata   criterio  cobertura
               ativo_total                                               Ativo Total nome exato     0.9187
        patrimonio_liquido                                        Patrimônio Líquido nome exato     0.9187
             lucro_liquido                                             Lucro Líquido nome exato     0.4375
resultado_antes_tributacao       Resultado antes da Tributação, Lucro e Participação nome exato     0.9187
     resultado_operacional                                     Resultado Operacional nome exato     0.9187
 resultado_nao_operacional                                 Resultado Não Operacional nome exato     0.9187
   resultado_intermediacao                     Resultado de Intermediação Financeira nome exato     0.9187
          provisao_credito Resultado de Provisão para Créditos de Difícil Liquidação nome exato     0.9187
       participacao_lucros           

---
## 4. Resolvendo o Lucro Líquido

O diagnóstico mostrou o problema com precisão: o rótulo `Lucro Líquido` existe em apenas 5 dos 12 exercícios, com 47,6% de cobertura, enquanto os **componentes da demonstração de resultado aparecem em 11 exercícios com 100%**. O BACEN mudou a estrutura do relatório ao longo do tempo e deixou de publicar a linha final em parte do período.

Há dois caminhos, e vale seguir os dois.

**Caminho principal, e o mais defensável tecnicamente.** Usar o **Resultado antes da Tributação** como base de rentabilidade em vez do lucro líquido. Cobertura de 100% em 11 anos, e conceitualmente é superior para previsão de insolvência: remove efeito fiscal e participações, que são decisões societárias e não medem geração de resultado. É o mesmo raciocínio pelo qual Altman usa EBIT sobre ativo em vez de lucro líquido sobre ativo.

**Caminho secundário, como validação.** Reconstruir o lucro líquido a partir dos componentes, subtraindo tributos e participação nos lucros, e conferir contra os 5 exercícios em que o valor publicado existe. Se a reconstrução bater, ela serve para a série inteira e vira um parágrafo forte na metodologia. Se não bater, fica registrado que não bate e usa-se só o caminho principal.

In [9]:
tem = lambda k: encontrados.get(k) is not None and k in base.columns

componentes = ["resultado_antes_tributacao", "tributos", "participacao_lucros",
               "resultado_operacional", "resultado_nao_operacional"]
print("componentes disponiveis:")
for c in componentes:
    print(f"   {c:<32}{'sim' if tem(c) else 'NAO'}   {encontrados.get(c) or ''}")

if tem("resultado_antes_tributacao"):
    parcelas = base["resultado_antes_tributacao"].copy()
    usados = ["resultado_antes_tributacao"]
    for comp in ["tributos", "participacao_lucros"]:
        if tem(comp):
            parcelas = parcelas - base[comp].fillna(0)
            usados.append(comp)
    base["lucro_liquido_reconstruido"] = parcelas
    print(f"\nreconstrucao feita com: {usados}")
else:
    base["lucro_liquido_reconstruido"] = np.nan
    print("\nresultado antes da tributacao ausente, reconstrucao impossivel")

componentes disponiveis:
   resultado_antes_tributacao      sim   Resultado antes da Tributação, Lucro e Participação
   tributos                        sim   Imposto de Renda e Contribuição Social
   participacao_lucros             sim   Participação nos Lucros
   resultado_operacional           sim   Resultado Operacional
   resultado_nao_operacional       sim   Resultado Não Operacional

reconstrucao feita com: ['resultado_antes_tributacao', 'tributos', 'participacao_lucros']


In [10]:
if tem("lucro_liquido"):
    comp = base[["ano", "lucro_liquido", "lucro_liquido_reconstruido", "ativo_total"]].dropna(
        subset=["lucro_liquido", "lucro_liquido_reconstruido"])
    if len(comp):
        comp["erro_relativo"] = ((comp["lucro_liquido_reconstruido"] - comp["lucro_liquido"]).abs()
                                 / comp["lucro_liquido"].abs().clip(lower=1))
        correl = comp["lucro_liquido"].corr(comp["lucro_liquido_reconstruido"])
        print(f"validacao em {len(comp):,} observacoes com os dois valores\n")
        print(f"correlacao de Pearson        {correl:.4f}")
        print(f"erro relativo mediano        {comp['erro_relativo'].median():.2%}")
        print(f"erro relativo no percentil 90 {comp['erro_relativo'].quantile(0.9):.2%}")
        print(f"observacoes com erro abaixo de 5%: {(comp['erro_relativo'] < 0.05).mean():.1%}")
        print("\npor exercicio:")
        print(comp.groupby("ano")["erro_relativo"].median().round(4).to_string())
        if correl > 0.98 and comp["erro_relativo"].median() < 0.05:
            print("\nVEREDITO: reconstrucao valida, pode ser usada na serie inteira.")
        else:
            print("\nVEREDITO: reconstrucao nao reproduz o publicado. Use resultado antes da tributacao.")
    else:
        print("nao ha observacoes com os dois valores para comparar")
else:
    print("lucro liquido publicado ausente, sem base de comparacao")

validacao em 7,242 observacoes com os dois valores

correlacao de Pearson        0.6465
erro relativo mediano        5.97%
erro relativo no percentil 90 129.45%
observacoes com erro abaixo de 5%: 47.6%

por exercicio:
ano
2014   0.0503
2015   0.0580
2016   0.0586
2017   0.0746
2018   0.0693

VEREDITO: reconstrucao nao reproduz o publicado. Use resultado antes da tributacao.


## 5. Indicadores reconstruídos

O ROA e o ROE passam a ter duas versões: a principal, sobre o resultado antes da tributação, com cobertura ampla, e a alternativa, sobre o lucro líquido reconstruído. Ter as duas permite reportar na metodologia que o resultado não depende dessa escolha, o que é exatamente o tipo de robustez que uma banca valoriza.

In [12]:
def razao(num, den, minimo=None, maximo=None):
    n = pd.to_numeric(base[num], errors="coerce") if isinstance(num, str) else num
    d = pd.to_numeric(base[den], errors="coerce") if isinstance(den, str) else den
    r = np.where((d.isna()) | (d <= 0) | (n.isna()), np.nan, n / d)
    r = pd.Series(r, index=base.index)
    if minimo is not None:
        r = r.where(r >= minimo)
    if maximo is not None:
        r = r.where(r <= maximo)
    return r

base["roa"] = razao("resultado_antes_tributacao", "ativo_total", -1, 1)
base["roe"] = razao("resultado_antes_tributacao", "patrimonio_liquido", -5, 5)
base["roa_lucro_liquido"] = razao("lucro_liquido_reconstruido", "ativo_total", -1, 1)
base["pl_sobre_ativo"] = razao("patrimonio_liquido", "ativo_total", -2, 1.5)
base["alavancagem"] = razao("ativo_total", "patrimonio_liquido", 0, 200)
base["credito_sobre_ativo"] = razao("carteira_credito", "ativo_total", 0, 1.5)
base["captacoes_sobre_ativo"] = razao("captacoes", "ativo_total", 0, 2)
if tem("provisao_credito"):
    base["provisao_sobre_credito"] = razao(base["provisao_credito"].abs(), "carteira_credito", 0, 1)
if tem("resultado_intermediacao"):
    base["margem_intermediacao"] = razao("resultado_intermediacao", "ativo_total", -1, 1)
if tem("rwa"):
    base["rwa_sobre_ativo"] = razao("rwa", "ativo_total", 0, 3)

for col in ["basileia", "capital_principal"]:
    if tem(col):
        mediana = base[col].median()
        if pd.notna(mediana) and mediana < 1.5:
            base[col] = base[col] * 100
        base[col] = base[col].where(base[col].between(0, 200))
        base[f"{col}_ausente"] = base[col].isna().astype("int8")

base = base.sort_values(["cod_inst", "ano"])
base["crescimento_ativo"] = base.groupby("cod_inst")["ativo_total"].pct_change().clip(-1, 5)
base["prejuizo"] = (base["resultado_antes_tributacao"] < 0).astype("int8")
base["log_ativo"] = np.log(base["ativo_total"].clip(lower=1))

IND_B = [c for c in ["roa", "roe", "roa_lucro_liquido", "pl_sobre_ativo", "alavancagem",
                     "basileia", "capital_principal", "credito_sobre_ativo",
                     "captacoes_sobre_ativo", "provisao_sobre_credito", "margem_intermediacao",
                     "rwa_sobre_ativo", "crescimento_ativo", "log_ativo"] if c in base.columns]

cob = pd.DataFrame({
    "cobertura": base[IND_B].notna().mean().round(4),
    "anos_com_dado": {c: int((base[c].groupby(base["ano"]).apply(lambda x: x.notna().mean()) > 0.2).sum())
                      for c in IND_B},
}).sort_values("cobertura", ascending=False)
print(cob.to_string())

                        cobertura  anos_com_dado
log_ativo                  0.9187             11
captacoes_sobre_ativo      0.9187             11
pl_sobre_ativo             0.9183             11
credito_sobre_ativo        0.9180             11
roa                        0.9167             11
roa_lucro_liquido          0.9157             11
alavancagem                0.9151             11
roe                        0.9141             11
margem_intermediacao       0.9117             11
crescimento_ativo          0.8811             11
basileia                   0.8682             11
rwa_sobre_ativo            0.7870             10
provisao_sobre_credito     0.7866             11
capital_principal          0.4574             11


In [13]:
print("cobertura por exercicio dos principais indicadores\n")
principais = [c for c in ["roa", "roe", "pl_sobre_ativo", "basileia", "capital_principal",
                          "credito_sobre_ativo"] if c in base.columns]
matriz = pd.DataFrame({c: base[c].groupby(base["ano"]).apply(lambda x: round(x.notna().mean(), 3))
                       for c in principais})
print(matriz.to_string())

cobertura por exercicio dos principais indicadores

        roa    roe  pl_sobre_ativo  basileia  capital_principal  credito_sobre_ativo
ano                                                                                 
2014 0.9980 0.9970          1.0000    0.0000             0.0000               0.9990
2015 0.9990 0.9950          0.9990    0.9590             0.9590               1.0000
2016 0.9990 0.9960          1.0000    0.9600             0.9600               1.0000
2017 0.9950 0.9920          0.9990    0.9600             0.9600               0.9960
2018 0.9980 0.9960          0.9990    0.9560             0.3190               0.9990
2019 0.9960 0.9950          0.9980    0.9790             0.3230               0.9980
2020 0.9980 0.9940          1.0000    0.9630             0.3270               0.9980
2021 0.9970 0.9940          0.9980    0.9570             0.3290               1.0000
2022 0.9970 0.9930          0.9990    0.9260             0.3150               0.9990
2023 0.9990 0

---
## 6. O rótulo de resolução

Este é o problema de verdade da camada B. Sem a lista oficial de instituições liquidadas não há variável resposta, e sem variável resposta não há estudo preditivo.

A dificuldade é que a lista do BACEN identifica instituições por CNPJ e nome, enquanto o IF.data usa um código próprio e não publica CNPJ no cadastro. Não existe chave comum, e o pareamento tem que ser por nome, com conferência humana.

O notebook monta o insumo dessa conferência: para cada evento da lista oficial, os três candidatos mais parecidos do IF.data, com um escore de similaridade que combina sobreposição de palavras e semelhança de caracteres. Você marca `S` nos corretos e o rótulo é aplicado.

In [15]:
CKAN = "https://dadosabertos.bcb.gov.br/api/3/action/package_show?id=instituicoes-submetidas-a-regimes-de-resolucao"

if not any(DIR_RES.glob("*")):
    try:
        import requests
        pacote = requests.get(CKAN, timeout=120).json()
        n = 0
        for r in pacote.get("result", {}).get("resources", []):
            fmt = (r.get("format") or "").upper()
            if fmt not in ("CSV", "XLSX", "XLS"):
                continue
            ext = {"CSV": ".csv", "XLSX": ".xlsx", "XLS": ".xls"}[fmt]
            nome = re.sub(r"[^A-Za-z0-9]+", "_", normalizar(r.get("name", "recurso")))[:60] + ext
            (DIR_RES / nome).write_bytes(requests.get(r["url"], timeout=300).content)
            n += 1
        print(f"{n} recursos baixados para {DIR_RES}")
    except Exception as e:
        print("download automatico falhou:", type(e).__name__, str(e)[:100])
        print("\nBaixe manualmente em:")
        print("https://dadosabertos.bcb.gov.br/dataset/instituicoes-submetidas-a-regimes-de-resolucao")
        print(f"e salve os CSV ou XLSX em {DIR_RES}")
else:
    print(f"{len(list(DIR_RES.glob('*')))} arquivos ja em {DIR_RES}")

for p in sorted(DIR_RES.glob("*")):
    print(f"   {p.name:<58}{p.stat().st_size/1000:,.1f} KB")

1 arquivos ja em G:\Meu Drive\TCC\Damaris Rebeca\01_bruto\bcb_resolucao
   INSTITUICOES_EM_LIQUIDACAO_EXTRAJUDICIAL.json             31.5 KB


In [16]:
partes = []
for p in sorted(DIR_RES.glob("*")):
    if p.suffix.lower() not in (".csv", ".xlsx", ".xls"):
        continue
    try:
        if p.suffix.lower() == ".csv":
            try:
                t = pd.read_csv(p, sep=";", encoding="latin-1", dtype=str)
            except Exception:
                t = pd.read_csv(p, sep=",", encoding="utf-8", dtype=str)
        else:
            t = pd.read_excel(p, dtype=str)
        t["recurso"] = p.name
        partes.append(t)
        print(f"{p.name:<50}{len(t):>6} linhas")
        print(f"   colunas: {list(t.columns)}")
    except Exception as e:
        print(f"{p.name}: ERRO {type(e).__name__}")

manual = DIR_INTER / "resolucao_manual.csv"
if manual.exists():
    t = pd.read_csv(manual, dtype=str)
    t["recurso"] = "manual"
    partes.append(t)
    print(f"{'resolucao_manual.csv':<50}{len(t):>6} linhas")

resolucao = pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()
print(f"\n{len(resolucao)} registros de resolucao carregados")
if len(resolucao):
    print(resolucao.head(5).to_string(max_colwidth=30))


0 registros de resolucao carregados


Se nada foi carregado, prepare o arquivo manual. Ele precisa de duas colunas apenas, `instituicao` e `ano_evento`, e pode ser montado a partir da página do BACEN em poucos minutos. Salve como `02_intermediario/resolucao_manual.csv` e rode a célula anterior de novo.

In [18]:
SUFIXOS = [r"\bS\.?A\.?\b", r"\bLTDA\.?\b", r"\bEIRELI\b", r"\bME\b", r"\bEPP\b",
           r"EM LIQUIDACAO EXTRAJUDICIAL", r"EM LIQUIDACAO", r"EM INTERVENCAO",
           r"\bRAET\b", r"\bEM RECUPERACAO\b", r"\bCIA\b", r"\bCOMPANHIA\b"]

def chave_nome(s):
    n = normalizar(s)
    for pad in SUFIXOS:
        n = re.sub(pad, " ", n)
    n = re.sub(r"[^A-Z0-9 ]", " ", n)
    return " ".join(n.split())

def similaridade(a, b):
    ta, tb = set(a.split()), set(b.split())
    if not ta or not tb:
        return 0.0
    jaccard = len(ta & tb) / len(ta | tb)
    sequencia = SequenceMatcher(None, a, b).ratio()
    return round(0.5 * jaccard + 0.5 * sequencia, 4)

catalogo_if = base.drop_duplicates(subset=["cod_inst"])[["cod_inst", "instituicao", "uf"]].copy()
catalogo_if["chave"] = catalogo_if["instituicao"].map(chave_nome)
catalogo_if = catalogo_if[catalogo_if["chave"].str.len() > 2]
print(f"{len(catalogo_if):,} instituicoes no catalogo do IF.data")
print(catalogo_if.head(5).to_string(index=False))

1,967 instituicoes no catalogo do IF.data
 cod_inst                                                                                                                                                   instituicao uf                                                                                                                                               chave
    52058                                                                                                                                                        NUBANK SP                                                                                                                                              NUBANK
    58338                                                                                                 COOPERATIVA DE CRÉDITO RURAL DE BOM JARDIM LTDA. - BOM CREDI. RJ                                                                                                COOPERATIVA DE CREDITO RURAL DE BOM JARDIM BOM CREDI
 

In [19]:
def coluna_por_termo(df, *termos):
    for t in termos:
        alvo = normalizar(t)
        for c in df.columns:
            if alvo in normalizar(c):
                return c
    return None

eventos = pd.DataFrame()
if len(resolucao):
    col_nome = coluna_por_termo(resolucao, "instituicao", "nome", "razao social", "entidade", "denominacao")
    col_data = coluna_por_termo(resolucao, "data", "decreta", "inicio", "publicacao")
    col_ano = coluna_por_termo(resolucao, "ano")
    col_regime = coluna_por_termo(resolucao, "regime", "tipo", "situacao", "modalidade")
    print(f"nome: {col_nome} | data: {col_data} | ano: {col_ano} | regime: {col_regime}")

    if col_nome:
        eventos = pd.DataFrame({"instituicao_evento": resolucao[col_nome]})
        if col_data:
            eventos["ano_evento"] = pd.to_datetime(resolucao[col_data], errors="coerce",
                                                   dayfirst=True).dt.year
        if col_ano:
            eventos["ano_evento"] = eventos.get("ano_evento", pd.Series(index=eventos.index, dtype=float)).fillna(
                pd.to_numeric(resolucao[col_ano], errors="coerce"))
        if col_regime:
            eventos["regime"] = resolucao[col_regime]
        eventos = eventos.dropna(subset=["instituicao_evento"]).drop_duplicates()
        eventos["chave"] = eventos["instituicao_evento"].map(chave_nome)

anos_base = sorted(base["ano"].unique())
if len(eventos):
    print(f"\n{len(eventos)} eventos | com ano valido: {int(eventos['ano_evento'].notna().sum())}")
    na_janela = eventos["ano_evento"].between(min(anos_base) + 1, max(anos_base) + 1)
    print(f"dentro da janela {min(anos_base)} a {max(anos_base)}: {int(na_janela.sum())}")
    print(eventos.head(10).to_string(index=False, max_colwidth=44))
else:
    print("\nnenhum evento estruturado. O pareamento ficara vazio.")


nenhum evento estruturado. O pareamento ficara vazio.


In [20]:
pareamento = pd.DataFrame()
if len(eventos):
    chaves_if = catalogo_if["chave"].tolist()
    linhas = []
    for _, e in eventos.iterrows():
        escores = [(c, similaridade(e["chave"], c)) for c in chaves_if]
        escores.sort(key=lambda x: -x[1])
        topo = escores[:3]
        registro = {"instituicao_evento": e["instituicao_evento"],
                    "ano_evento": e.get("ano_evento"),
                    "regime": e.get("regime", "")}
        for i, (chave, escore) in enumerate(topo, start=1):
            linha_if = catalogo_if[catalogo_if["chave"] == chave].iloc[0]
            registro[f"cand{i}"] = linha_if["instituicao"]
            registro[f"cod{i}"] = linha_if["cod_inst"]
            registro[f"score{i}"] = escore
        registro["confirmado"] = "S" if topo and topo[0][1] >= SIMILARIDADE_MINIMA else ""
        registro["cod_inst"] = topo[0][1] >= SIMILARIDADE_MINIMA and registro.get("cod1") or None
        linhas.append(registro)

    pareamento = pd.DataFrame(linhas).sort_values("score1", ascending=False)
    pareamento.to_csv(DIR_INTER / "pareamento_resolucao.csv", index=False, encoding="utf-8-sig")

    auto = int((pareamento["confirmado"] == "S").sum())
    print(f"{len(pareamento)} eventos pareados | {auto} acima do limiar de {SIMILARIDADE_MINIMA}")
    print(f"\narquivo para conferencia: {DIR_INTER / 'pareamento_resolucao.csv'}")
    print("\ntop 20 por escore:\n")
    cols = ["instituicao_evento", "ano_evento", "cand1", "score1", "confirmado"]
    print(pareamento[cols].head(20).to_string(index=False, max_colwidth=40))

### Conferência

Abra `02_intermediario/pareamento_resolucao.csv`. Cada linha traz o evento oficial e os três candidatos mais próximos do IF.data com seus escores. Marque `S` na coluna `confirmado` e copie o código correto para `cod_inst` nas linhas certas, apague a marcação nas erradas.

Escore acima de 0,90 costuma ser correspondência exata com diferença de grafia. Entre 0,80 e 0,90 exige olhar. Abaixo de 0,80 geralmente é instituição que nunca esteve no IF.data, o que é comum: muita instituição liquidada é pequena demais para figurar nos relatórios de conglomerados prudenciais.

Depois de conferir, rode a célula abaixo com `USAR_CONFERENCIA = True`.

In [22]:
USAR_CONFERENCIA = False

if USAR_CONFERENCIA and (DIR_INTER / "pareamento_resolucao.csv").exists():
    pareamento = pd.read_csv(DIR_INTER / "pareamento_resolucao.csv")
    print("pareamento recarregado do arquivo conferido")

eventos_validos = pd.DataFrame()
if len(pareamento):
    ok = pareamento[(pareamento["confirmado"].astype(str).str.upper().str.strip() == "S")
                    & pareamento["cod_inst"].notna()
                    & pareamento["ano_evento"].notna()].copy()
    ok["cod_inst"] = pd.to_numeric(ok["cod_inst"], errors="coerce")
    ok["ano_evento"] = pd.to_numeric(ok["ano_evento"], errors="coerce").astype("Int64")
    eventos_validos = ok.dropna(subset=["cod_inst"]).drop_duplicates(subset=["cod_inst"])

print(f"{len(eventos_validos)} eventos confirmados para rotulagem")
if len(eventos_validos):
    print(eventos_validos[["instituicao_evento", "ano_evento", "cand1", "score1"]].to_string(
        index=False, max_colwidth=40))

0 eventos confirmados para rotulagem


---
## 7. Rótulos alternativos

Além da lista oficial, duas fontes de evento que não custam nada e podem complementar a contagem.

**Carimbo no nome.** O cadastro do IF.data às vezes traz "EM LIQUIDAÇÃO EXTRAJUDICIAL" na própria razão social, do mesmo modo que o cadastro da CVM. É o truque que funcionou na camada A.

**Desaparecimento da base.** Uma instituição presente até certo exercício e ausente depois saiu do sistema. Pode ter sido liquidada, incorporada ou ter mudado de segmento. **Isso não é rótulo de insolvência**, e usar assim seria erro grave. Serve como lista de verificação: cruzar esses desaparecimentos com a lista oficial ajuda a encontrar eventos que o pareamento por nome deixou passar.

In [25]:
carimbo = base["instituicao"].map(normalizar).str.contains(
    r"EM LIQUIDACAO|EM INTERVENCAO|LIQUIDACAO EXTRAJUDICIAL|RAET", regex=True, na=False)
por_carimbo = (base[carimbo].sort_values("ano").drop_duplicates(subset=["cod_inst"])
                 [["cod_inst", "instituicao", "ano", "uf"]]
                 .rename(columns={"ano": "primeiro_ano_com_carimbo"}))
print(f"{len(por_carimbo)} instituicoes com carimbo de regime no nome")
if len(por_carimbo):
    print(por_carimbo.to_string(index=False, max_colwidth=50))

0 instituicoes com carimbo de regime no nome


In [26]:
ultimo_ano = base.groupby("cod_inst")["ano"].max()
primeiro_ano = base.groupby("cod_inst")["ano"].min()
ano_final = max(base["ano"])

saiu = ultimo_ano[ultimo_ano < ano_final - 1]
saidas = pd.DataFrame({"cod_inst": saiu.index, "ultimo_ano": saiu.values})
saidas = saidas.merge(base.drop_duplicates(subset=["cod_inst"])[["cod_inst", "instituicao", "uf"]],
                      on="cod_inst", how="left")
saidas["primeiro_ano"] = saidas["cod_inst"].map(primeiro_ano)
saidas["anos_na_base"] = saidas["ultimo_ano"] - saidas["primeiro_ano"] + 1
saidas = saidas.sort_values("ultimo_ano", ascending=False)
saidas.to_csv(DIR_INTER / "saidas_ifdata.csv", index=False, encoding="utf-8-sig")

print(f"{len(saidas):,} instituicoes deixaram a base antes do ultimo exercicio")
print("\nsaidas por ano:")
print(saidas["ultimo_ano"].value_counts().sort_index().to_string())
print(f"\nlista completa em {DIR_INTER / 'saidas_ifdata.csv'}")
print("\nATENCAO: saida da base nao e evidencia de liquidacao. Serve so para conferencia cruzada.")

545 instituicoes deixaram a base antes do ultimo exercicio

saidas por ano:
ultimo_ano
2014    77
2015    64
2016    70
2017    52
2018    68
2019    39
2020    39
2021    37
2022    58
2023    41

lista completa em G:\Meu Drive\TCC\Damaris Rebeca\02_intermediario\saidas_ifdata.csv

ATENCAO: saida da base nao e evidencia de liquidacao. Serve so para conferencia cruzada.


---
## 8. Aplicação do rótulo e balanço

O rótulo vai no exercício **anterior** ao evento, mantendo o desenho de previsão com defasagem de um ano usado nas outras camadas. Exercícios a partir do evento são marcados para descarte, porque um balanço já sob liquidação não prevê nada, apenas registra o que aconteceu.

In [28]:
INCLUIR_CARIMBO = True

base["y_resolucao"] = 0
base["y_resolucao_h2"] = 0
base["descartar_pos_evento"] = 0
base["origem_evento"] = ""

def aplicar(cod, ano, origem):
    m = base["cod_inst"] == cod
    base.loc[m & (base["ano"] == ano - 1), "y_resolucao"] = 1
    base.loc[m & base["ano"].between(ano - 2, ano - 1), "y_resolucao_h2"] = 1
    base.loc[m & (base["ano"] >= ano), "descartar_pos_evento"] = 1
    base.loc[m, "origem_evento"] = origem

aplicados = 0
for _, ev in eventos_validos.iterrows():
    aplicar(int(ev["cod_inst"]), int(ev["ano_evento"]), "lista oficial")
    aplicados += 1

if INCLUIR_CARIMBO:
    ja = set(eventos_validos["cod_inst"].astype(int)) if len(eventos_validos) else set()
    for _, ev in por_carimbo.iterrows():
        if int(ev["cod_inst"]) in ja:
            continue
        aplicar(int(ev["cod_inst"]), int(ev["primeiro_ano_com_carimbo"]), "carimbo no nome")
        aplicados += 1

elegivel = base[base["descartar_pos_evento"] == 0]
pos1 = int(elegivel["y_resolucao"].sum())
pos2 = int(elegivel["y_resolucao_h2"].sum())

print(f"{aplicados} eventos aplicados\n")
print(f"observacoes totais      {len(base):>8,}")
print(f"elegiveis               {len(elegivel):>8,}")
print(f"positivos horizonte 1   {pos1:>8,}  ({pos1/max(len(elegivel),1):.3%})")
print(f"positivos horizonte 2   {pos2:>8,}  ({pos2/max(len(elegivel),1):.3%})")
if aplicados:
    print("\npor origem:")
    print(base[base["y_resolucao"] == 1]["origem_evento"].value_counts().to_string())

0 eventos aplicados

observacoes totais        16,552
elegiveis                 16,552
positivos horizonte 1          0  (0.000%)
positivos horizonte 2          0  (0.000%)


---
## 9. Veredito

O critério é simples e foi fixado antes de olhar o resultado, que é como se toma decisão metodológica sem enviesar a si mesma. Trinta casos positivos é o piso usual para estimar um classificador com validação estratificada sem que cada partição fique com um punhado de eventos.

In [32]:
cobertura_media = float(base[[c for c in ["roa", "pl_sobre_ativo", "basileia"] if c in base.columns]].notna().mean().mean())

print("=" * 68)
print("VEREDITO DA CAMADA B")
print("=" * 68)
print(f"observacoes elegiveis           {len(elegivel):>8,}")
print(f"instituicoes                    {base['cod_inst'].nunique():>8,}")
print(f"exercicios                      {min(base['ano'])} a {max(base['ano'])}")
print(f"positivos horizonte 1           {pos1:>8,}")
print(f"positivos horizonte 2           {pos2:>8,}")
print(f"cobertura media dos indicadores {cobertura_media:>8.1%}")
print("-" * 68)

melhor = max(pos1, pos2)
if melhor >= MIN_POSITIVOS_MODELAGEM and cobertura_media >= 0.7:
    veredito = "ESTUDO PREDITIVO"
    texto = ("A camada B sustenta modelo proprio. Use o horizonte com mais positivos, "
             "amostra pareada por porte e segmento, e reporte o horizonte alternativo como robustez.")
elif melhor >= 10:
    veredito = "PREDITIVO FRAGIL"
    texto = ("Ha eventos, mas poucos. Caminhos: ampliar a coleta para frequencia trimestral, "
             "o que multiplica por quatro as observacoes e capta eventos que o corte anual perde, "
             "ou tratar a camada como descritiva. Nao apresente metricas preditivas com essa contagem "
             "sem intervalo de confianca e sem declarar a limitacao.")
else:
    veredito = "DESCRITIVA"
    texto = ("Sem eventos suficientes. A camada B entra no artigo como caracterizacao do sistema "
             "financeiro, com Basileia, rentabilidade e concentracao, servindo de contraponto as "
             "demais camadas. A previsao de resolucao vira trabalho futuro, com justificativa "
             "empirica registrada e nao como promessa vaga.")

print(f"{veredito}\n")
print(texto)
print("=" * 68)

VEREDITO DA CAMADA B
observacoes elegiveis             16,552
instituicoes                       1,967
exercicios                      2014 a 2025
positivos horizonte 1                  0
positivos horizonte 2                  0
cobertura media dos indicadores    90.1%
--------------------------------------------------------------------
DESCRITIVA

Sem eventos suficientes. A camada B entra no artigo como caracterizacao do sistema financeiro, com Basileia, rentabilidade e concentracao, servindo de contraponto as demais camadas. A previsao de resolucao vira trabalho futuro, com justificativa empirica registrada e nao como promessa vaga.


In [33]:
CHAVES_SAIDA = [c for c in ["ano", "data_base", "tipo_instituicao", "cod_inst", "instituicao",
                            "tcb", "consolidacao", "controle", "uf", "cidade", "segmento",
                            "origem_evento", "descartar_pos_evento", "fonte"] if c in base.columns]
BRUTOS = [c for c in ["ativo_total", "patrimonio_liquido", "resultado_antes_tributacao",
                      "lucro_liquido", "lucro_liquido_reconstruido", "carteira_credito",
                      "captacoes", "prejuizo"] if c in base.columns]
AUSENCIA = [c for c in base.columns if c.endswith("_ausente")]
ALVOS = ["y_resolucao", "y_resolucao_h2"]

camada_b = base[CHAVES_SAIDA + BRUTOS + IND_B + AUSENCIA + ALVOS].copy()
camada_b.to_parquet(DIR_MODELAGEM / "camada_b_recuperada.parquet", index=False)
print(f"camada_b_recuperada: {len(camada_b):,} linhas x {camada_b.shape[1]} colunas")

mg = camada_b[camada_b["uf"].fillna("").str.upper().str.strip() == UF_RECORTE]
mg.to_parquet(DIR_MODELAGEM / f"camada_b_recuperada_{UF_RECORTE.lower()}.parquet", index=False)
print(f"camada_b_recuperada_{UF_RECORTE.lower()}: {len(mg):,} linhas | "
      f"{mg['cod_inst'].nunique()} instituicoes | {int(mg['y_resolucao'].sum())} positivos")

resumo = {
    "gerado_em": datetime.now().isoformat(timespec="seconds"),
    "veredito": veredito,
    "observacoes": int(len(camada_b)),
    "elegiveis": int(len(elegivel)),
    "instituicoes": int(camada_b["cod_inst"].nunique()),
    "exercicios": [int(min(base["ano"])), int(max(base["ano"]))],
    "positivos_h1": pos1,
    "positivos_h2": pos2,
    "cobertura_media_indicadores": round(cobertura_media, 4),
    "indicadores": IND_B,
    "mapeamento": {k: v for k, v in encontrados.items()},
    "eventos_lista_oficial": int(len(eventos_validos)),
    "eventos_carimbo": int(len(por_carimbo)),
}
(DIR_MODELAGEM / "resumo_camada_b.json").write_text(
    json.dumps(resumo, ensure_ascii=False, indent=2), encoding="utf-8")
print("\n" + json.dumps(resumo, ensure_ascii=False, indent=2)[:1800])

camada_b_recuperada: 16,552 linhas x 40 colunas
camada_b_recuperada_mg: 2,478 linhas | 254 instituicoes | 0 positivos

{
  "gerado_em": "2026-09-10T00:36:01",
  "veredito": "DESCRITIVA",
  "observacoes": 16552,
  "elegiveis": 16552,
  "instituicoes": 1967,
  "exercicios": [
    2014,
    2025
  ],
  "positivos_h1": 0,
  "positivos_h2": 0,
  "cobertura_media_indicadores": 0.9011,
  "indicadores": [
    "roa",
    "roe",
    "roa_lucro_liquido",
    "pl_sobre_ativo",
    "alavancagem",
    "basileia",
    "capital_principal",
    "credito_sobre_ativo",
    "captacoes_sobre_ativo",
    "provisao_sobre_credito",
    "margem_intermediacao",
    "rwa_sobre_ativo",
    "crescimento_ativo",
    "log_ativo"
  ],
  "mapeamento": {
    "ativo_total": "Ativo Total",
    "patrimonio_liquido": "Patrimônio Líquido",
    "lucro_liquido": "Lucro Líquido",
    "resultado_antes_tributacao": "Resultado antes da Tributação, Lucro e Participação",
    "resultado_operacional": "Resultado Operacional",
    "r

A camada A tem 48 eventos e sustenta o estudo principal de insolvência. A camada C tem 1,4 milhão de empresas mineiras e sustenta o estudo de mortalidade. A camada B vira uma seção de caracterização do sistema financeiro, que ainda assim enriquece o artigo, e a previsão de resolução vira trabalho futuro com justificativa empírica.

In [51]:
import requests

SERVICO = "https://olinda.bcb.gov.br/olinda/servico/regimes_especiais/versao/v1/odata"

def tenta(url, timeout=90):
    try:
        r = requests.get(url, timeout=timeout)
        return r.status_code, r
    except Exception as e:
        print(f"   erro: {type(e).__name__}: {str(e)[:90]}")
        return None, None

print("documento de servico")
st, r = tenta(f"{SERVICO}/?$format=json")
print("   status:", st)

colecoes = []
if r is not None and st == 200:
    try:
        colecoes = [v.get("name") or v.get("url") for v in r.json().get("value", [])]
    except Exception:
        colecoes = list(dict.fromkeys(re.findall(r'"(?:name|url)"\s*:\s*"([A-Za-z0-9_]+)"', r.text)))
        colecoes += list(dict.fromkeys(re.findall(r'metadata:name="([A-Za-z0-9_]+)"', r.text)))
    print("   colecoes:", colecoes)
else:
    print("   servico nao respondeu, tentando nomes prováveis")

if not colecoes:
    colecoes = ["Regimes", "RegimesEspeciais", "InstituicoesEmLiquidacao",
                "ListaRegimes", "_Regimes", "Instituicoes"]

print("\ntestando cada colecao")
validas = []
for nome in colecoes:
    st, r = tenta(f"{SERVICO}/{nome}?$format=json&$top=3", timeout=60)
    n = 0
    if st == 200:
        try:
            n = len(r.json().get("value", []))
        except Exception:
            n = 0
    print(f"   {nome:<28}status {st}  registros na amostra: {n}")
    if n:
        validas.append(nome)
        print("      colunas:", list(r.json()["value"][0].keys()))

print("\ncolecoes validas:", validas)

documento de servico
   status: 200
   colecoes: ['Regimes']

testando cada colecao
   Regimes                     status 200  registros na amostra: 3
      colunas: ['CnpjInstituicao', 'CnpjRaiz', 'NomeInstituicao', 'Inicio', 'Tipo', 'Responsavel', 'Endereco', 'Municipio', 'UF', 'CEP', 'Emails', 'Telefones']

colecoes validas: ['Regimes']


In [53]:
COLECAO = validas[0] if validas else None

if COLECAO:
    todos, skip = [], 0
    while True:
        st, r = tenta(f"{SERVICO}/{COLECAO}?$format=json&$top=1000&$skip={skip}", timeout=120)
        if st != 200:
            break
        bloco = r.json().get("value", [])
        if not bloco:
            break
        todos.extend(bloco)
        if len(bloco) < 1000:
            break
        skip += 1000

    regimes = pd.DataFrame(todos)
    destino = DIR_RES / "regimes_especiais.csv"
    regimes.to_csv(destino, sep=";", index=False, encoding="latin-1")

    print(f"{len(regimes):,} registros | colunas: {list(regimes.columns)}")
    print(f"gravado em {destino}\n")
    print(regimes.head(10).to_string(max_colwidth=34))

    for c in regimes.columns:
        if any(t in normalizar(c) for t in ["DATA", "ANO"]):
            anos = pd.to_datetime(regimes[c], errors="coerce").dt.year.dropna()
            if len(anos):
                print(f"\n{c}: de {int(anos.min())} a {int(anos.max())}")
                print(anos[anos.between(2014, 2025)].value_counts().sort_index().to_string())
else:
    print("Nenhuma colecao respondeu.")
    print("Plano B: abra https://www.bcb.gov.br/estabilidadefinanceira/regimesespeciais")
    print("copie a tabela para uma planilha com as colunas instituicao e ano_evento,")
    print(f"e salve como {DIR_INTER / 'resolucao_manual.csv'}")

23 registros | colunas: ['CnpjInstituicao', 'CnpjRaiz', 'NomeInstituicao', 'Inicio', 'Tipo', 'Responsavel', 'Endereco', 'Municipio', 'UF', 'CEP', 'Emails', 'Telefones']
gravado em G:\Meu Drive\TCC\Damaris Rebeca\01_bruto\bcb_resolucao\regimes_especiais.csv

  CnpjInstituicao  CnpjRaiz                    NomeInstituicao      Inicio                      Tipo                        Responsavel                           Endereco       Municipio  UF        CEP                             Emails                 Telefones
0  33171211000146  33171211  ACQIO ADQUIRENCIA INSTITUICAO ...  27/03/2026  LIQUIDAÇÃO EXTRAJUDICIAL              CASSIO HAIG VARTANIAN      Av. Paulista, 475 - 12º andar       São Paulo  SP  01311-000            liquidacao@acqio.com.br              (11)40001829
1  92856905000186  92856905  ADVANCED CORRETORA DE CÂMBIO LTDA  15/01/2026  LIQUIDAÇÃO EXTRAJUDICIAL              FABIANO FABRI BAYARRI        Rua Salvador Simões, nº 801       São Paulo  SP  04276-000  liquidacao@ad